# K-Nearest Neighbors — Tanítás

**Felelős:** Kasza Dominik Árpád

Nemparametrikus **KNN regresszor** (`KNeighborsRegressor`), StandardScaler-rel Pipeline-ba csomagolva (a feldolgozó pipeline nem tartalmaz skálázást, KNN viszont távolságon alapul, ezért kötelező). A hiperparamétereket (`n_neighbors`, `weights`, `metric`) GridSearchCV-vel hangoljuk.

**Skálázhatósági korlát:** KNN előrejelzési ideje O(n × d), ahol n a tanítóminták száma. A teljes 7,6M soros tanítóhalmazon a predikció órákat venne igénybe, ezért:
- **Hangolás:** `TUNE_SAMPLE_SIZE = 10 000` soros sample (GridSearchCV, cv=5)
- **Végső modell:** `TRAIN_SAMPLE_SIZE = 50 000` soros sample (refitelt modell, Drive-ra mentve)

Mentett kimenetek (Drive: `MODELS_DIR`):
- `knn_tuned.joblib` — a hangolt Pipeline (StandardScaler + KNN)
- `knn_predictions_test.npy` — test predikciók
- `knn_validation_curve_n_neighbors.npz` — validation curve arrays
- `knn_learning_curve.npz` — learning curve arrays

Plus `results/metrics.csv` (Drive) — append-elt metrika sorok.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Slityak/gepitan-seoul-bike-trip/blob/main/notebooks/06_knn.ipynb)

## 1. Setup (Colab + lokális kompatibilis)

In [1]:
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
BRANCH = 'main'

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    if not os.path.exists('/content/gepitan-beadando'):
        !git clone --branch {BRANCH} https://github.com/Slityak/gepitan-seoul-bike-trip.git /content/gepitan-beadando

    os.chdir('/content/gepitan-beadando')
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
    !pip install -q -r requirements.txt
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')

sys.path.insert(0, '.')
print(f'Working dir: {os.getcwd()}')
print(f'In Colab: {IN_COLAB}')
print(f'Branch: {BRANCH}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
From https://github.com/Slityak/gepitan-seoul-bike-trip
 * branch            main       -> FETCH_HEAD
Already on 'main'
Your branch is up to date with 'origin/main'.
From https://github.com/Slityak/gepitan-seoul-bike-trip
 * branch            main       -> FETCH_HEAD
Already up to date.
Working dir: /content/gepitan-beadando
In Colab: True
Branch: main


In [2]:
import numpy as np
import joblib
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, validation_curve, learning_curve

from src.config import RANDOM_SEED, CV_FOLDS, MODELS_DIR, ensure_dirs
from src.data_io import load_v1_data
from src.evaluation import append_metrics, evaluate_model

ensure_dirs()
MODELS_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
from src import config
from pathlib import Path

MY_OUTPUT_DIR = Path('/content/drive/MyDrive/knn_output')
MY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(MY_OUTPUT_DIR / 'figures').mkdir(exist_ok=True)

config.MODELS_DIR  = MY_OUTPUT_DIR
config.RESULTS_DIR = MY_OUTPUT_DIR
config.FIGURES_DIR = MY_OUTPUT_DIR / 'figures'
config.METRICS_CSV = MY_OUTPUT_DIR / 'metrics.csv'

# Shortcut változó az explicit átadáshoz
METRICS_OUT = config.METRICS_CSV

print(f'Kimenetek ide kerülnek: {MY_OUTPUT_DIR}')


Kimenetek ide kerülnek: /content/drive/MyDrive/knn_output


## 2. Adatbetöltés

In [4]:
X_train, X_test, y_train, y_test, feature_names = load_v1_data()

print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')
print(f'Target — train: mean={y_train.mean():.2f}, std={y_train.std():.2f}, '
      f'min={y_train.min()}, max={y_train.max()}')

# KNN-hez mintázott teszthalmaz — predikció O(n_train × n_test), a teljes 1.9M sor túl lassú
TEST_SAMPLE_SIZE = 50_000
rng_test = np.random.default_rng(RANDOM_SEED)
test_idx = rng_test.choice(len(X_test), size=min(TEST_SAMPLE_SIZE, len(X_test)), replace=False)
X_test_knn = X_test[test_idx]
y_test_knn = y_test[test_idx]
print(f'KNN test sample: {X_test_knn.shape}')

np.save(MY_OUTPUT_DIR / 'knn_test_idx.npy', test_idx)



Train shape: (7680911, 36), Test shape: (1920228, 36)
Target — train: mean=25.80, std=25.04, min=1, max=119
KNN test sample: (50000, 36)


## 3. Baseline: alap KNN (no tuning)

Default `KNeighborsRegressor(n_neighbors=5)` StandardScaler-rel — referenciapont a hangolt változathoz.

A skálázás kötelező: az adatpipeline (`geostat_elemzes.py`) nem tartalmaz StandardScaler-t (a HGBR-hez nincs rá szükség), KNN viszont euklideszi távolságon alapul, ahol a feature skálák közvetlenül befolyásolják a szomszédok keresését.

In [5]:
DEFAULT_SAMPLE_SIZE = 20_000

rng_default = np.random.default_rng(RANDOM_SEED)
idx_default = rng_default.choice(len(X_train), size=min(DEFAULT_SAMPLE_SIZE, len(X_train)), replace=False)
X_default_sample = X_train[idx_default]
y_default_sample = y_train[idx_default]

knn_pipeline_default = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsRegressor(n_neighbors=5, n_jobs=-1)),
])

result_default, _ = evaluate_model(
    knn_pipeline_default,
    X_default_sample, y_default_sample,
    X_test_knn, y_test_knn,
    model_name='KNN (default)',
    notes=f'n_neighbors=5, uniform, euclidean, fit on n={DEFAULT_SAMPLE_SIZE}, eval on n={TEST_SAMPLE_SIZE}',
)

print(f'MAE: {result_default.mae:.3f}')
print(f'RMSE: {result_default.rmse:.3f}')
print(f'R²: {result_default.r2:.3f}')
print(f'Train idő: {result_default.train_time_sec:.2f} s')
print(f'Predict idő: {result_default.predict_time_sec:.2f} s')

append_metrics(result_default, csv_path=METRICS_OUT)


MAE: 15.004
RMSE: 22.239
R²: 0.209
Train idő: 0.01 s
Predict idő: 11.41 s


## 4. GridSearchCV hangolás (sample-en) + refit

1. `TUNE_SAMPLE_SIZE = 10 000` soros random sample a train-ből → ezen GridSearchCV (cv=5)
2. A legjobb paraméterekkel refit `TRAIN_SAMPLE_SIZE = 50 000` soros sample-en
   (a teljes 7,6M soros adaton a KNN predikció impraktikus lenne)
3. Mentés: Pipeline (scaler + knn) + test predikciók + metrika

A param_grid 8 kombinációt fed le (4 × 2 × 1), ami 40 CV fit-et jelent.

In [6]:
TUNE_SAMPLE_SIZE = 10_000
TRAIN_SAMPLE_SIZE = 50_000

rng_tune = np.random.default_rng(RANDOM_SEED)
tune_idx = rng_tune.choice(len(X_train), size=min(TUNE_SAMPLE_SIZE, len(X_train)), replace=False)
X_tune, y_tune = X_train[tune_idx], y_train[tune_idx]

param_grid = {
    'knn__n_neighbors': [5, 10, 20, 50],
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ['euclidean'],
}

knn_pipeline_base = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsRegressor(n_jobs=-1)),
])

grid = GridSearchCV(
    estimator=knn_pipeline_base,
    param_grid=param_grid,
    cv=CV_FOLDS,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1,
)
grid.fit(X_tune, y_tune)

print(f'\nTune sample size: {len(X_tune)}')
print(f'Legjobb paraméterek: {grid.best_params_}')
print(f'CV legjobb score (neg MAE): {grid.best_score_:.3f}')

rng_train = np.random.default_rng(RANDOM_SEED + 1)
train_idx = rng_train.choice(len(X_train), size=min(TRAIN_SAMPLE_SIZE, len(X_train)), replace=False)
X_train_sample = X_train[train_idx]
y_train_sample = y_train[train_idx]

best_params = {
    k.replace('knn__', ''): v for k, v in grid.best_params_.items()
}
best_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsRegressor(**best_params, n_jobs=-1)),
])

result_tuned, y_pred_tuned = evaluate_model(
    best_pipeline,
    X_train_sample, y_train_sample,
    X_test_knn, y_test_knn,
    model_name='KNN (tuned)',
    notes=f'GridSearchCV cv={CV_FOLDS} on n={TUNE_SAMPLE_SIZE}, refit on n={TRAIN_SAMPLE_SIZE}; '
          f'best: {grid.best_params_}',
)

print(f'\nTeszt MAE: {result_tuned.mae:.3f}')
print(f'Teszt RMSE: {result_tuned.rmse:.3f}')
print(f'Teszt R²: {result_tuned.r2:.3f}')
print(f'Train idő: {result_tuned.train_time_sec:.2f} s')
print(f'Predict idő: {result_tuned.predict_time_sec:.2f} s')

append_metrics(result_tuned, csv_path=METRICS_OUT)

joblib.dump(best_pipeline, MY_OUTPUT_DIR / 'knn_tuned.joblib')
np.save(MY_OUTPUT_DIR / 'knn_predictions_test.npy', y_pred_tuned)
np.save(MY_OUTPUT_DIR / 'knn_y_test_sample.npy', y_test_knn)
print(f'Mentve: {MY_OUTPUT_DIR}')



Fitting 5 folds for each of 8 candidates, totalling 40 fits

Tune sample size: 10000
Legjobb paraméterek: {'knn__metric': 'euclidean', 'knn__n_neighbors': 20, 'knn__weights': 'distance'}
CV legjobb score (neg MAE): -14.692

Teszt MAE: 13.916
Teszt RMSE: 20.850
Teszt R²: 0.305
Train idő: 0.05 s
Predict idő: 21.02 s
Mentve: /content/drive/MyDrive/knn_output


## 5. Validation curve (n_neighbors)

Megmutatja, hogy az `n_neighbors` változtatása hogyan hat a CV MAE-re — ugyanúgy mint a Decision Tree max_depth görbéje. Kis n → overfit (train MAE alacsony, CV MAE magas), nagy n → underfit (mindkettő magas).

In [ ]:
DIAG_SAMPLE_SIZE = 30_000

diag_rng = np.random.default_rng(RANDOM_SEED)
diag_idx = diag_rng.choice(len(X_train), size=min(DIAG_SAMPLE_SIZE, len(X_train)), replace=False)
X_diag, y_diag = X_train[diag_idx], y_train[diag_idx]
print(f'Diagnosztikai sample: {X_diag.shape}')

# A legjobb weights és metric fixálva, n_neighbors változtatjuk
best_weights = grid.best_params_.get('knn__weights', 'distance')
best_metric = grid.best_params_.get('knn__metric', 'euclidean')

vc_param_range = [3, 5, 10, 20, 50, 100, 200]

vc_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsRegressor(weights=best_weights, metric=best_metric, n_jobs=-1)),
])

vc_train_scores, vc_test_scores = validation_curve(
    vc_pipeline,
    X_diag, y_diag,
    param_name='knn__n_neighbors',
    param_range=vc_param_range,
    cv=CV_FOLDS,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
)
vc_train_mae = -vc_train_scores
vc_test_mae = -vc_test_scores

vc_labels = np.array([str(k) for k in vc_param_range])
np.savez(
    MODELS_DIR / 'knn_validation_curve_n_neighbors.npz',
    param_range_labels=vc_labels,
    train_mae=vc_train_mae,
    test_mae=vc_test_mae,
    best_weights=best_weights,
    best_metric=best_metric,
)
print(f'Validation curve mentve: {MODELS_DIR}/knn_validation_curve_n_neighbors.npz')
print(f'(weights={best_weights}, metric={best_metric})')
for lbl, tr, te in zip(vc_labels, vc_train_mae.mean(axis=1), vc_test_mae.mean(axis=1)):
    print(f'  n_neighbors={lbl:>4s}: train MAE={tr:.3f}, CV MAE={te:.3f}')

Diagnosztikai sample: (30000, 36)
Validation curve mentve: /content/drive/MyDrive/knn_output/knn_validation_curve_n_neighbors.npz
(weights=distance, metric=euclidean)
  n_neighbors=   3: train MAE=0.000, CV MAE=15.167
  n_neighbors=   5: train MAE=0.000, CV MAE=14.538
  n_neighbors=  10: train MAE=0.000, CV MAE=14.060
  n_neighbors=  20: train MAE=0.000, CV MAE=13.900
  n_neighbors=  50: train MAE=0.000, CV MAE=14.085
  n_neighbors= 100: train MAE=0.000, CV MAE=14.430
  n_neighbors= 200: train MAE=0.000, CV MAE=14.807


## 6. Learning curve

Megmutatja, hogy a tuned KNN mennyi adattal közelíti a CV-plateaut — analóg a Decision Tree learning curve-jével.

In [ ]:
lc_train_sizes = np.linspace(0.1, 1.0, 8)
lc_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsRegressor(**best_params, n_jobs=-1)),
])

lc_sizes, lc_train_scores, lc_test_scores = learning_curve(
    lc_pipeline,
    X_diag, y_diag,
    train_sizes=lc_train_sizes,
    cv=CV_FOLDS,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    shuffle=True,
    random_state=RANDOM_SEED,
)
lc_train_mae = -lc_train_scores
lc_test_mae = -lc_test_scores

np.savez(
    MODELS_DIR / 'knn_learning_curve.npz',
    sizes=lc_sizes,
    train_mae=lc_train_mae,
    test_mae=lc_test_mae,
)
print(f'Learning curve mentve: {MODELS_DIR}/knn_learning_curve.npz')
for sz, tr, te in zip(lc_sizes, lc_train_mae.mean(axis=1), lc_test_mae.mean(axis=1)):
    print(f'  size={int(sz):>6d}: train MAE={tr:.3f}, CV MAE={te:.3f}')

Learning curve mentve: /content/drive/MyDrive/knn_output/knn_learning_curve.npz
  size=  2400: train MAE=0.000, CV MAE=15.088
  size=  5485: train MAE=0.000, CV MAE=14.738
  size=  8571: train MAE=0.000, CV MAE=14.434
  size= 11657: train MAE=0.000, CV MAE=14.290
  size= 14742: train MAE=0.000, CV MAE=14.120
  size= 17828: train MAE=0.000, CV MAE=14.051
  size= 20914: train MAE=0.000, CV MAE=13.958
  size= 24000: train MAE=0.000, CV MAE=13.900


## 7. Tanulságok

Kiemelendő, hogy a KNN futási ideje miatt az összes számítás mintavételezett adatokon történt.
_A számokat itt érdemes összegezni; a vizualizációk a `06_knn_visualize.ipynb`-ben._

- A legjobb `n_neighbors`, `weights` és `metric` értékek a GridSearchCV alapján: lásd fentebb.
- A KNN a baseline modelleknél (mean/median) és tipikusan a Ridge regressziónál is jobb, de a döntési fa alulmúlhatja — a nemlineáris struktúra felfogásában a fa hatékonyabb.
- **Skálázhatóság:** a KNN predikciós ideje O(n × d), ezért a teljes 7,6M soros trainon nem praktikus; 50k soros mintán refitelt modell kompromisszumot jelent teljesítmény és sebesség között.